<a href="https://colab.research.google.com/github/sreevarshini22/CODSOFT/blob/main/Task3/Churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from sklearn.pipeline import Pipeline


# SECTION 1 - DATA LOADING


def load_dataset(filepath="Churn_Modelling.csv"):
    """
    Loads the Kaggle Bank Customer Churn dataset.
    If the file is not found, generates equivalent synthetic data
    using the exact same columns and distributions.

    Parameters
    ----------
    filepath : str
        Path to Churn_Modelling.csv

    Returns
    -------
    pd.DataFrame
    """
    if os.path.exists(filepath):
        print(f"            Loading real dataset from: {filepath}")
        df = pd.read_csv(filepath)
        # Drop identifier columns not useful for modeling
        df = df.drop(columns=["RowNumber", "CustomerId", "Surname"], errors="ignore")
        return df
    else:
        print("            Churn_Modelling.csv not found.")
        print("            Generating synthetic data with identical columns...")
        return generate_synthetic_bank_data()


def generate_synthetic_bank_data(n=10000, seed=42):
    """
    Generates synthetic data that mirrors the Kaggle Churn_Modelling.csv
    schema exactly: same columns, same data types, similar distributions.

    This allows the code to run and be demonstrated without needing
    the original file. Replace with the real CSV when available.
    """
    rng = np.random.default_rng(seed)

    credit_score      = rng.integers(350, 851, n)
    geography         = rng.choice(["France", "Germany", "Spain"], n, p=[0.50, 0.25, 0.25])
    gender            = rng.choice(["Male", "Female"], n, p=[0.545, 0.455])
    age               = rng.integers(18, 93, n)
    tenure            = rng.integers(0, 11, n)
    balance           = np.where(
        rng.random(n) < 0.36, 0.0,
        rng.uniform(11000, 250900, n)
    ).round(2)
    num_of_products   = rng.choice([1, 2, 3, 4], n, p=[0.46, 0.46, 0.06, 0.02])
    has_cr_card       = rng.choice([0, 1], n, p=[0.29, 0.71])
    is_active_member  = rng.choice([0, 1], n, p=[0.49, 0.51])
    estimated_salary  = rng.uniform(11, 199992, n).round(2)

    # Churn probability based on known drivers in this dataset
    churn_prob = (
        0.08
        + 0.10 * (geography == "Germany")
        + 0.05 * (gender == "Female")
        + 0.004 * np.clip(age - 36, 0, 50)
        - 0.04 * is_active_member
        + 0.06 * (num_of_products >= 3)
        - 0.02 * (balance == 0)
        + 0.001 * (850 - credit_score) / 10
    )
    churn_prob = np.clip(churn_prob, 0.03, 0.90)
    exited     = rng.binomial(1, churn_prob).astype(int)

    return pd.DataFrame({
        "CreditScore":     credit_score,
        "Geography":       geography,
        "Gender":          gender,
        "Age":             age,
        "Tenure":          tenure,
        "Balance":         balance,
        "NumOfProducts":   num_of_products,
        "HasCrCard":       has_cr_card,
        "IsActiveMember":  is_active_member,
        "EstimatedSalary": estimated_salary,
        "Exited":          exited,
    })

# SECTION 2 - EXPLORATORY DATA ANALYSIS


def print_eda_summary(df):
    """Prints a concise summary of the loaded dataset."""
    print(f"\n            Shape         : {df.shape}")
    print(f"            Churn Rate    : {df['Exited'].mean():.2%}")
    print(f"            Missing Values: {df.isnull().sum().sum()}")
    print(f"            Columns       : {list(df.columns)}")



# SECTION 3 - PREPROCESSING


def prepare_data(raw_df):
    """
    Encodes categorical features and splits into train/test sets.

    Steps:
        1. Label-encode Geography and Gender
        2. Separate features from target (Exited)
        3. Stratified 80/20 split

    Returns
    -------
    tuple : X_train, X_test, y_train, y_test, feature_names
    """
    df        = raw_df.copy()
    label_enc = LabelEncoder()

    for col in ["Geography", "Gender"]:
        if col in df.columns:
            df[col] = label_enc.fit_transform(df[col])

    X          = df.drop("Exited", axis=1)
    y          = df["Exited"]
    feat_names = X.columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    return X_train, X_test, y_train, y_test, feat_names


# SECTION 4 - MODEL DEFINITIONS


def define_classifiers():
    """
    Returns three classifiers configured for the bank churn dataset.

    - Logistic Regression: scaled pipeline, L2 regularization
    - Random Forest      : 300 trees, balanced class weight for imbalance
    - Gradient Boosting  : 200 estimators, low learning rate for stability
    """
    lr = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=1.0, max_iter=1000, solver="lbfgs", random_state=42
        ))
    ])

    rf = RandomForestClassifier(
        n_estimators=300, max_depth=10,
        min_samples_leaf=4, class_weight="balanced",
        random_state=42, n_jobs=-1
    )

    gb = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05,
        max_depth=4, subsample=0.8, random_state=42
    )

    return {
        "Logistic Regression": lr,
        "Random Forest":       rf,
        "Gradient Boosting":   gb,
    }



# SECTION 5 - EVALUATION


def evaluate_model(model, X_test, y_test, name="Model"):
    """
    Scores a trained model and prints a full report.

    Returns
    -------
    tuple : (score_dict, y_pred, y_proba)
    """
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    scores = {
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall":    recall_score(y_test, y_pred, zero_division=0),
        "F1 Score":  f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_test, y_proba),
    }

    print("\n" + "-" * 54)
    print("  " + name)
    print("-" * 54)
    for k, v in scores.items():
        print(f"  {k:<12} : {v:.4f}")
    print()
    print(classification_report(y_test, y_pred,
                                target_names=["Retained", "Churned"]))
    return scores, y_pred, y_proba


# SECTION 6 - VISUALIZATION DASHBOARD

BG      = "#0D1117"
CARD    = "#161B22"
TEXT    = "#E6EDF3"
GRID    = "#21262D"
BLUE    = "#388BFD"
ORANGE  = "#F78166"
GREEN   = "#3FB950"
CLRS    = [BLUE, ORANGE, GREEN]


def style(ax, title="", xlabel="", ylabel=""):
    ax.set_facecolor(CARD)
    ax.tick_params(colors=TEXT, labelsize=8)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    ax.title.set_color(TEXT)
    if title:   ax.set_title(title, fontsize=10, fontweight="bold", pad=8)
    if xlabel:  ax.set_xlabel(xlabel, fontsize=8)
    if ylabel:  ax.set_ylabel(ylabel, fontsize=8)
    ax.grid(axis="y", color=GRID, linewidth=0.5, alpha=0.6)


def build_dashboard(results, clf_map, X_tr, X_te, y_tr, y_te,
                    feat_names, raw_df, out_path):

    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "text.color":  TEXT,
        "axes.facecolor":   CARD,
        "figure.facecolor": BG,
        "savefig.facecolor": BG,
    })

    fig = plt.figure(figsize=(24, 28))
    fig.patch.set_facecolor(BG)
    fig.text(0.5, 0.987, "BANK CUSTOMER CHURN PREDICTION",
             ha="center", fontsize=22, fontweight="bold", color=TEXT)
    fig.text(0.5, 0.977, "Kaggle Dataset  |  Logistic Regression  |  Random Forest  |  Gradient Boosting",
             ha="center", fontsize=11, color="#8B949E")

    gs       = gridspec.GridSpec(5, 4, figure=fig, hspace=0.55, wspace=0.40,
                                 left=0.06, right=0.97, top=0.965, bottom=0.03)
    names    = list(results.keys())
    m_keys   = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]

    # ---- Row 0: churn distribution | metric bars | geography churn | AUC bar ----
    ax0 = fig.add_subplot(gs[0, 0])
    vc  = raw_df["Exited"].value_counts()
    b   = ax0.bar(["Retained", "Churned"], vc.values,
                  color=[BLUE, ORANGE], edgecolor=BG, width=0.5)
    for bar, v in zip(b, vc.values):
        ax0.text(bar.get_x() + bar.get_width()/2, v + 60, str(v),
                 ha="center", color=TEXT, fontsize=9, fontweight="bold")
    style(ax0, "Class Distribution", "", "Count")
    ax0.grid(False)

    ax1 = fig.add_subplot(gs[0, 1:3])
    x   = np.arange(len(m_keys))
    w   = 0.22
    for i, (nm, (mets, _, _)) in enumerate(results.items()):
        ax1.bar(x + i*w, [mets[k] for k in m_keys], width=w,
                label=nm, color=CLRS[i], edgecolor=BG, alpha=0.92)
    ax1.set_xticks(x + w)
    ax1.set_xticklabels(m_keys, fontsize=8, color=TEXT)
    ax1.set_ylim(0, 1.12)
    ax1.legend(fontsize=8, facecolor=CARD, edgecolor=GRID, labelcolor=TEXT)
    style(ax1, "Model Performance Comparison", "", "Score")

    ax2 = fig.add_subplot(gs[0, 3])
    auc_v = [results[n][0]["ROC-AUC"] for n in names]
    hb    = ax2.barh(names, auc_v, color=CLRS, edgecolor=BG)
    for bar, v in zip(hb, auc_v):
        ax2.text(v - 0.05, bar.get_y() + bar.get_height()/2,
                 f"{v:.3f}", va="center", color="white",
                 fontsize=9, fontweight="bold")
    ax2.set_xlim(0, 1)
    style(ax2, "ROC-AUC Summary", "AUC", "")
    ax2.grid(axis="x", color=GRID, linewidth=0.5, alpha=0.6)
    ax2.grid(False, axis="y")

    # ---- Row 1: ROC curves | Precision-Recall curves ----
    ax3 = fig.add_subplot(gs[1, 0:2])
    for i, (nm, (mets, _, proba)) in enumerate(results.items()):
        fpr, tpr, _ = roc_curve(y_te, proba)
        ax3.plot(fpr, tpr, color=CLRS[i], lw=2,
                 label=f"{nm}  (AUC={mets['ROC-AUC']:.3f})")
    ax3.plot([0,1],[0,1], "--", color=GRID, lw=1)
    ax3.legend(fontsize=8, facecolor=CARD, edgecolor=GRID, labelcolor=TEXT)
    style(ax3, "ROC Curves", "False Positive Rate", "True Positive Rate")

    ax4 = fig.add_subplot(gs[1, 2:4])
    for i, (nm, (_, _, proba)) in enumerate(results.items()):
        p, r, _ = precision_recall_curve(y_te, proba)
        ax4.plot(r, p, color=CLRS[i], lw=2, label=nm)
    ax4.legend(fontsize=8, facecolor=CARD, edgecolor=GRID, labelcolor=TEXT)
    style(ax4, "Precision-Recall Curves", "Recall", "Precision")

    # ---- Row 2: Confusion matrices | Cross-validation ----
    for i, (nm, (_, preds, _)) in enumerate(results.items()):
        axc = fig.add_subplot(gs[2, i])
        cm  = confusion_matrix(y_te, preds)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    ax=axc, cbar=False, linewidths=0.5,
                    annot_kws={"size": 11, "color": "white"},
                    xticklabels=["Retained", "Churned"],
                    yticklabels=["Retained", "Churned"])
        axc.set_facecolor(CARD)
        axc.tick_params(colors=TEXT, labelsize=7.5)
        for sp in axc.spines.values():
            sp.set_edgecolor(GRID)
        axc.set_title("Confusion Matrix\n" + nm,
                      fontsize=9, fontweight="bold", color=TEXT)
        axc.set_xlabel("Predicted", color=TEXT, fontsize=8)
        axc.set_ylabel("Actual",    color=TEXT, fontsize=8)

    axcv = fig.add_subplot(gs[2, 3])
    cv   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for i, nm in enumerate(names):
        sc = cross_val_score(clf_map[nm], X_tr, y_tr,
                             cv=cv, scoring="roc_auc", n_jobs=-1)
        axcv.errorbar(i, sc.mean(), yerr=sc.std(),
                      fmt="o", color=CLRS[i], capsize=5, markersize=8,
                      label=f"{nm.split()[0]}\n{sc.mean():.3f}+/-{sc.std():.3f}")
    axcv.set_xticks(range(len(names)))
    axcv.set_xticklabels([n.split()[0] for n in names], fontsize=8, color=TEXT)
    axcv.set_ylim(0.6, 1.0)
    style(axcv, "5-Fold CV ROC-AUC", "", "AUC")

    # ---- Row 3: Feature importance RF | GB ----
    axfi1 = fig.add_subplot(gs[3, 0:2])
    rf_m  = clf_map["Random Forest"]
    pd.Series(rf_m.feature_importances_, index=feat_names)\
      .nlargest(10).sort_values()\
      .plot(kind="barh", ax=axfi1, color=ORANGE, edgecolor=BG)
    style(axfi1, "Random Forest - Top 10 Feature Importances", "Score", "")
    axfi1.tick_params(axis="y", labelsize=8)
    axfi1.grid(axis="x", color=GRID, linewidth=0.5, alpha=0.6)
    axfi1.grid(False, axis="y")

    axfi2 = fig.add_subplot(gs[3, 2:4])
    gb_m  = clf_map["Gradient Boosting"]
    pd.Series(gb_m.feature_importances_, index=feat_names)\
      .nlargest(10).sort_values()\
      .plot(kind="barh", ax=axfi2, color=GREEN, edgecolor=BG)
    style(axfi2, "Gradient Boosting - Top 10 Feature Importances", "Score", "")
    axfi2.tick_params(axis="y", labelsize=8)
    axfi2.grid(axis="x", color=GRID, linewidth=0.5, alpha=0.6)
    axfi2.grid(False, axis="y")

    # ---- Row 4: Probability distribution | Score card ----
    axpd  = fig.add_subplot(gs[4, 0:2])
    gp    = results["Gradient Boosting"][2]
    axpd.hist(gp[y_te == 0], bins=40, alpha=0.7, color=BLUE,
              label="Retained", edgecolor=BG)
    axpd.hist(gp[y_te == 1], bins=40, alpha=0.7, color=ORANGE,
              label="Churned",  edgecolor=BG)
    axpd.axvline(0.5, color=TEXT, lw=1.5, linestyle="--",
                 label="Threshold = 0.5")
    axpd.legend(fontsize=8, facecolor=CARD, edgecolor=GRID, labelcolor=TEXT)
    style(axpd, "Gradient Boosting - Predicted Probability Distribution",
          "Churn Probability", "Count")

    axtb  = fig.add_subplot(gs[4, 2:4])
    axtb.set_facecolor(CARD)
    axtb.axis("off")
    rows = [[mk] + [f"{results[n][0][mk]:.4f}" for n in names]
            for mk in m_keys]
    tbl  = axtb.table(cellText=rows,
                      colLabels=["Metric"] + names,
                      cellLoc="center", loc="center",
                      bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_facecolor(CARD if r % 2 == 0 else "#1C2128")
        cell.set_edgecolor(GRID)
        cell.set_text_props(
            color=ORANGE if r == 0 else TEXT,
            fontweight="bold" if r == 0 else "normal"
        )
    axtb.set_title("Final Score Card", fontsize=10,
                   fontweight="bold", color=TEXT, pad=8)

    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=BG)
    print(f"            Dashboard saved to: {out_path}")
    plt.close()


# SECTION 7 - MAIN


if __name__ == "__main__":

    OUT_IMG   = "/mnt/user-data/outputs/bank_churn_dashboard.png"
    OUT_MODEL = "/mnt/user-data/outputs/best_bank_churn_model.pkl"

    print("=" * 60)
    print("   BANK CUSTOMER CHURN PREDICTION")
    print("   Dataset: Kaggle - Churn_Modelling.csv")
    print("=" * 60)

    # Step 1 - Load data
    print("\n[Step 1/4]  Loading dataset...")
    df = load_dataset("Churn_Modelling.csv")
    print_eda_summary(df)

    # Step 2 - Preprocess
    print("\n[Step 2/4]  Preprocessing...")
    X_train, X_test, y_train, y_test, feat_cols = prepare_data(df)
    print(f"            Train : {X_train.shape[0]} records")
    print(f"            Test  : {X_test.shape[0]}  records")
    print(f"            Features used: {feat_cols}")

    # Step 3 - Train and evaluate
    print("\n[Step 3/4]  Training classifiers...")
    clf_definitions = define_classifiers()
    fitted_models   = {}
    all_results     = {}

    for clf_name, clf_obj in clf_definitions.items():
        print(f"\n            Fitting {clf_name}...")
        clf_obj.fit(X_train, y_train)
        fitted_models[clf_name] = clf_obj
        sc, pr, pb = evaluate_model(clf_obj, X_test, y_test, name=clf_name)
        all_results[clf_name] = (sc, pr, pb)

    # Step 4 - Dashboard
    print("\n[Step 4/4]  Rendering dashboard...")
    build_dashboard(
        results=all_results,
        clf_map=fitted_models,
        X_tr=X_train, X_te=X_test,
        y_tr=y_train, y_te=y_test,
        feat_names=feat_cols,
        raw_df=df,
        out_path=OUT_IMG
    )

    # Save best model
    best_name = max(all_results, key=lambda n: all_results[n][0]["ROC-AUC"])
    best_auc  = all_results[best_name][0]["ROC-AUC"]
    joblib.dump(fitted_models[best_name], OUT_MODEL)

    print("\n" + "=" * 60)
    print(f"   Best Model  : {best_name}")
    print(f"   ROC-AUC     : {best_auc:.4f}")
    print(f"   Model saved : {OUT_MODEL}")
    print("=" * 60)

   BANK CUSTOMER CHURN PREDICTION
   Dataset: Kaggle - Churn_Modelling.csv

[Step 1/4]  Loading dataset...
            Loading real dataset from: Churn_Modelling.csv

            Shape         : (10000, 11)
            Churn Rate    : 20.37%
            Missing Values: 0
            Columns       : ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']

[Step 2/4]  Preprocessing...
            Train : 8000 records
            Test  : 2000  records
            Features used: ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']

[Step 3/4]  Training classifiers...

            Fitting Logistic Regression...

------------------------------------------------------
  Logistic Regression
------------------------------------------------------
  Accuracy     : 0.8050
  Precision    : 0.5859
  Recall       : 0.1425
  F1 Score  